<a href="https://colab.research.google.com/github/ridamumtazz/Flyrank-ML-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Two Paper Findings + My Methodology Questions

### Finding 1: Content Lifecycle — Growing vs. Declining Content

The research report compares content that is growing with content that is declining and uses these groups to study differences in content performance. My methodology question is: how exactly were the growing and declining labels created, and what time period was used to define them? I would want to confirm that the label was based on a clearly defined future or comparison window and that the variables used to measure the outcome were not also used as inputs. This would help make sure the finding is measuring a real observed pattern rather than a result created by the way the label was constructed.

### Finding 2: Freshness Multiplier

The report observes that refreshing existing content is associated with improved performance and presents this as a useful content strategy. My methodology question is whether the validation design can distinguish an association from a causal effect. For example, content that is refreshed may also receive other improvements, such as better optimization or stronger promotion. It would therefore be useful to know whether the analysis compares refreshed content with a suitable baseline or control group and whether the validation design supports the claim that freshness itself caused the improvement. If the design is observational, the result is best described as a measured association rather than proof of causation.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## My Model Under an Honest Split

My Week-5 model used a time-aware split, where earlier observations were used for training and later observations were used for testing. This is an honest validation design for predicting next-period clicks because the model is evaluated on a later time period than the training data.

For the validation audit, I will also test the model using a grouped-by-client split. This provides an additional stress test by evaluating whether the model can generalize to clients that were not included in training. The original time-aware result is treated as the before result, while the grouped-by-client result provides the after comparison.

The original time-aware split produced an MAE of 0.262 and an RMSE of 2.4575. The grouped split will be evaluated using the same features, Random Forest configuration, and metrics so that the comparison is consistent. Any difference between the two results is treated as evidence about generalization, not as proof that one validation method is universally better.


In [ ]:
# ============================================
# ML-09 - PREPARE DATA FOR VALIDATION AUDIT
# ============================================

from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd
import numpy as np

# 1. Get Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

# 2. Download dataset
sample_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

# 3. Load dataset
sample_df = pd.read_parquet(sample_path)

print("Dataset is ready.")
print("Rows:", sample_df.shape[0])
print("Columns:", sample_df.shape[1])

# 4. Make a copy
model_df = sample_df.copy()

# 5. Convert date
model_df["report_date"] = pd.to_datetime(
    model_df["report_date"]
)

# 6. Create CTR
model_df["ctr"] = np.where(
    model_df["gsc_impressions"] > 0,
    model_df["gsc_clicks"] / model_df["gsc_impressions"],
    0
)

# 7. Create engagement rate
model_df["engagement_rate"] = np.where(
    model_df["ga4_sessions"] > 0,
    model_df["ga4_engaged_sessions"] / model_df["ga4_sessions"],
    0
)

# 8. Sort by content and date
model_df = model_df.sort_values(
    ["content_hash_id", "report_date"]
)

# 9. Create next-period clicks as target
model_df["target_clicks"] = (
    model_df
    .groupby("content_hash_id")["gsc_clicks"]
    .shift(-1)
)

# 10. Select features
features = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "ga4_sessions",
    "engagement_rate"
]

# 11. Remove missing values
model_df = model_df.dropna(
    subset=features + ["target_clicks"]
)

print("================================")
print("DATA PREPARATION COMPLETE")
print("================================")
print("Prepared data shape:", model_df.shape)
print("Features:", features)

In [10]:
# ============================================
# ML-09 SECTION 2 - GROUPED CLIENT SPLIT
# ============================================

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# Grouped split by client
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        groups=model_df["client_hash_id"]
    )
)

group_train_df = model_df.iloc[train_idx]
group_test_df = model_df.iloc[test_idx]

print("Grouped training rows:", len(group_train_df))
print("Grouped testing rows:", len(group_test_df))

# Check that clients do not overlap
train_clients = set(group_train_df["client_hash_id"])
test_clients = set(group_test_df["client_hash_id"])

print("Overlapping clients:", len(train_clients.intersection(test_clients)))

# Sample training data for manageable runtime
group_train_sample = group_train_df.sample(
    n=min(100000, len(group_train_df)),
    random_state=42
)

# Sample testing data
group_test_sample = group_test_df.sample(
    n=min(30000, len(group_test_df)),
    random_state=42
)

X_train_group = group_train_sample[features]
y_train_group = group_train_sample["target_clicks"]

X_test_group = group_test_sample[features]
y_test_group = group_test_sample["target_clicks"]

# Train the same Random Forest configuration
rf_group = RandomForestRegressor(
    n_estimators=50,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf_group.fit(
    X_train_group,
    y_train_group
)

# Predictions
group_predictions = rf_group.predict(X_test_group)

# Metrics
group_mae = mean_absolute_error(
    y_test_group,
    group_predictions
)

group_rmse = np.sqrt(
    mean_squared_error(
        y_test_group,
        group_predictions
    )
)

print("\n================================")
print("GROUPED CLIENT RESULTS")
print("================================")
print("MAE:", round(group_mae, 4))
print("RMSE:", round(group_rmse, 4))

# Before / After comparison
comparison = pd.DataFrame({
    "Validation": [
        "Before: Time-aware split",
        "After: Grouped-by-client split"
    ],
    "MAE": [
        0.262,
        round(group_mae, 4)
    ],
    "RMSE": [
        2.4575,
        round(group_rmse, 4)
    ]
})

display(comparison)

Grouped training rows: 1673815
Grouped testing rows: 1146016
Overlapping clients: 0

GROUPED CLIENT RESULTS
MAE: 0.5129
RMSE: 47.0867


,Validation,MAE,RMSE
0,Before: Time-aware split,0.2620,2.4575
1,After: Grouped-by-client split,0.5129,47.0867


### Before/After Interpretation

The original time-aware validation produced an MAE of 0.2620 and an RMSE of 2.4575. Under the grouped-by-client validation, the MAE increased to 0.5129 and the RMSE increased substantially to 47.0867. The grouped split had zero overlapping clients between training and testing, so it provides a stricter test of generalization to unseen clients.

The higher errors show that the model performs less consistently when evaluated on clients that were not represented in training. This suggests that the original time-aware result may be more representative of performance on known clients over future periods, while the grouped result highlights weaker cross-client generalization. The results do not prove that the model is unusable, but they show that validation design has a meaningful effect on the measured performance. Therefore, the model should be treated as directional decision-support, and its performance should not be assumed to generalize equally to all clients.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage Audit

I audited the final feature set for three main types of leakage: label-derived features, future or overlapping information, and decision-derived features.

The final features used by the Random Forest were `gsc_impressions`, `gsc_clicks`, `ctr`, `gsc_avg_position`, `ga4_sessions`, and `engagement_rate`. The model did not use `trend_direction`, `trend_pct`, `is_declining_label`, `action_score`, `action`, or `reason_code` as input features. These columns were excluded because they may contain information derived from labels, previous decisions, or existing scoring rules.

The target was created using the next available `gsc_clicks` value for each content item. Therefore, the model is intended to use the current observation to predict a later observation. The time-aware split also ensures that earlier dates are used for training and later dates for testing. However, the exact time gap between the current observation and the next observation should be checked because the next available row is not necessarily exactly one day later.

The `ctr` feature is calculated from `gsc_clicks` and `gsc_impressions`. This is not direct label leakage because the target is next-period clicks, but it is a derived feature that overlaps with information already present in two other features. Its relatively low feature importance of 0.0096 suggests that it provides limited additional signal in this model.

The grouped-by-client validation produced zero overlapping clients between training and testing. Its MAE increased from 0.2620 to 0.5129, while RMSE increased from 2.4575 to 47.0867. This indicates weaker generalization to unseen clients and shows why validation design matters when interpreting model performance.

Overall, I did not identify a direct label-derived or decision-derived feature in the final feature set. The main remaining validation risk is whether all feature values are strictly available before the target observation and whether the next-period target has a consistent time interval. These checks should be completed before treating the model as fully deployment-ready.


In [11]:
# ============================================
# ML-09 SECTION 3 - LEAKAGE CHECK
# ============================================

# Final model features
print("Final model features:")
for feature in features:
    print("-", feature)

# Columns that should NOT be model features
risky_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "action_score",
    "action",
    "reason_code",
    "content_hash_id",
    "client_hash_id"
]

used_risky = [
    col for col in risky_columns
    if col in features
]

print("\nRisky columns used as features:")
print(used_risky)

if len(used_risky) == 0:
    print("PASS: No known risky columns are used as model features.")
else:
    print("WARNING: Review these columns for possible leakage.")

Final model features:
- gsc_impressions
- gsc_clicks
- ctr
- gsc_avg_position
- ga4_sessions
- engagement_rate

Risky columns used as features:
[]
PASS: No known risky columns are used as model features.


In [12]:
# ============================================
# CHECK CLIENT OVERLAP
# ============================================

train_clients = set(
    group_train_df["client_hash_id"]
)

test_clients = set(
    group_test_df["client_hash_id"]
)

overlap = train_clients.intersection(
    test_clients
)

print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Overlapping clients:", len(overlap))

if len(overlap) == 0:
    print("PASS: No clients appear in both training and testing.")
else:
    print("WARNING: Client overlap detected.")

Training clients: 39
Testing clients: 10
Overlapping clients: 0
PASS: No clients appear in both training and testing.


The grouped validation split contained 39 training clients and 10 testing clients, with zero clients shared between the two sets. This confirms that the grouped split was implemented correctly and that the reported test performance measures generalization to unseen clients rather than performance on clients already represented in training.


In [13]:
timing_df = model_df[
    [
        "content_hash_id",
        "report_date",
        "target_clicks"
    ]
].copy()

timing_df["next_report_date"] = (
    timing_df
    .groupby("content_hash_id")["report_date"]
    .shift(-1)
)

timing_df["days_to_next_observation"] = (
    timing_df["next_report_date"]
    - timing_df["report_date"]
).dt.days

print("Time gap between observations:")
display(
    timing_df["days_to_next_observation"].describe()
)

print("\nMost common time gaps:")
display(
    timing_df["days_to_next_observation"]
    .value_counts()
    .head(10)
)

Time gap between observations:


,days_to_next_observation
count,2.666243e+06
mean,1.204075e+00
std,1.070347e+00
min,0.000000e+00
25%,1.000000e+00
50%,1.000000e+00
75%,1.000000e+00
max,2.800000e+01



Most common time gaps:


,count
days_to_next_observation,
1.0,2453761
2.0,109457
3.0,39692
4.0,19910
5.0,11984
6.0,7789
7.0,5780
8.0,3565
9.0,2744


### Target Timing Check

The target was created using the next available `gsc_clicks` observation for each content item. The timing audit shows that the median gap between the current observation and the target observation is 1 day, and 75% of the gaps are also 1 day. The mean gap is approximately 1.20 days. However, the gap ranges from 0 to 28 days, meaning that the target is not consistently a fixed one-day-ahead prediction for every row.

This is an important limitation of the current target definition. Most observations represent approximately next-day clicks, but some predictions use a later observation because the next available record may be several days away. The zero-day gaps also require further investigation because they may indicate multiple observations for the same content on the same date.

Therefore, the model should be described as predicting the next available observed click value rather than strictly predicting next-day clicks. A more rigorous future version could define a fixed prediction horizon and explicitly align the feature and target windows.


In [14]:
# ============================================
# CHECK SAME-DAY DUPLICATES
# ============================================

duplicate_dates = (
    model_df
    .groupby(
        ["content_hash_id", "report_date"]
    )
    .size()
    .reset_index(name="row_count")
)

same_day_duplicates = duplicate_dates[
    duplicate_dates["row_count"] > 1
]

print(
    "Content-date combinations with duplicates:",
    len(same_day_duplicates)
)

print("\nTop duplicate counts:")
display(
    same_day_duplicates
    .sort_values(
        "row_count",
        ascending=False
    )
    .head(10)
)

Content-date combinations with duplicates: 1429

Top duplicate counts:


,content_hash_id,report_date,row_count
2813548,content_ff881e76e85001f4,2026-06-22,2
9459,content_00e045559f33deac,2026-06-17,2
9460,content_00e045559f33deac,2026-06-18,2
9461,content_00e045559f33deac,2026-06-19,2
9462,content_00e045559f33deac,2026-06-20,2
2764436,content_fb1b290d16ca0d29,2026-06-24,2
2764435,content_fb1b290d16ca0d29,2026-06-22,2
2764434,content_fb1b290d16ca0d29,2026-06-21,2
2755580,content_fa60e9dea96f683a,2026-06-24,2
2755579,content_fa60e9dea96f683a,2026-06-22,2


### Duplicate-Date Check

The timing audit identified 1,429 content-date combinations with duplicate rows. This explains the observed zero-day gaps between the current observation and the target observation. Because the target was created using `shift(-1)` after sorting by content and date, duplicate rows on the same date can cause the target to come from another row with the same reporting date rather than from a genuinely future observation.

This does not prove that the model has direct label leakage, but it is a weakness in the current target construction. The target is intended to represent a later observation, but the current implementation does not guarantee that the target is strictly in the future for every row. A more rigorous approach would first resolve duplicate content-date records and then create a target using a strictly later date or a clearly defined future prediction window.

Therefore, the current results should be interpreted cautiously. The model provides directional decision-support based on the available data, but the target construction should be improved before making stronger claims about future performance.


In [15]:
# ============================================
# ML-09 SECTION 3 - REAL FAILURE EXAMPLES
# ============================================

error_df = group_test_sample[
    [
        "content_hash_id",
        "client_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "ga4_sessions",
        "target_clicks"
    ]
].copy()

error_df["predicted_clicks"] = group_predictions

error_df["absolute_error"] = (
    np.abs(
        error_df["target_clicks"]
        - error_df["predicted_clicks"]
    )
)

largest_errors = (
    error_df
    .sort_values(
        "absolute_error",
        ascending=False
    )
    .head(10)
)

print("Top 10 largest prediction errors:")
display(largest_errors)

Top 10 largest prediction errors:


,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_clicks,ga4_sessions,target_clicks,predicted_clicks,absolute_error
3815548,content_adcc7b85a04c187d,client_9c26c096d6e57253,2026-06-07,12667,7106,15786.0,8329.0,180.460000,8148.540000
10388672,content_bbff2da5b5610190,client_73cda7b4e4f265ea,2026-06-26,9654,457,4790.0,506.0,181.140000,324.860000
8960195,content_6503c571b86265df,client_fef1a8f436438636,2026-06-26,999,48,609.0,62.0,33.660000,28.340000
11576203,content_73b528f19edba3ba,client_3f0ce4d44fe94f3d,2026-06-27,512,15,149.0,0.0,13.322857,13.322857
4358630,content_6c4da03688ca8351,client_73cda7b4e4f265ea,2026-06-10,3268,21,22.0,11.0,22.520000,11.520000
4708636,content_5267d90f451c6edc,client_73cda7b4e4f265ea,2026-06-13,1263,9,10.0,19.0,7.571770,11.428230
759372,content_18effdf03138153d,client_73cda7b4e4f265ea,2026-06-01,2504,25,26.0,33.0,21.680000,11.320000
1378083,content_ef7013c86d07aa99,client_e5c2aa26a8598242,2026-06-04,5310,29,25.0,36.0,24.940000,11.060000
7890200,content_e92b45fa68bef0df,client_73cda7b4e4f265ea,2026-06-18,372,1,1.0,12.0,1.354630,10.645370
5042171,content_0cd251309ac4bcb2,client_73cda7b4e4f265ea,2026-06-15,1088,7,4.0,16.0,6.152126,9.847874


### Failure Examples and Interpretation

The failure analysis shows that the model can make very large errors for some observations. The largest error in the grouped test sample occurred for one content item where the observed target was 8,329 clicks while the model predicted approximately 180 clicks, producing an absolute error of about 8,148.54 clicks. Other examples also show smaller but noticeable differences, such as predicting 181 clicks for an observed target of 506 clicks.

These examples help explain why the grouped-by-client RMSE of 47.0867 is much higher than the MAE of 0.5129. The large difference suggests that a small number of observations have unusually large prediction errors. The model appears to struggle when future click performance changes sharply compared with the current observed signals.

The failure examples also show why the model should not be treated as a guaranteed predictor of future clicks. Instead, the results should be used as directional decision-support for prioritizing content, with large-error cases and unusual traffic changes treated cautiously.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim Rewrite

### Original Claim

The Random Forest model can predict future content performance and help identify which content should be improved first.

### Rewritten Claim

In this dataset, the Random Forest produced measured predictions of the next available observed click value using search and engagement signals. The model showed lower error under the time-aware validation split than under the grouped-by-client split, suggesting that performance is weaker when generalizing to unseen clients. The results provide directional decision-support for prioritizing content, but they do not demonstrate that the model will generalize equally to all clients or guarantee future click performance. The target construction and duplicate content-date records should also be improved before making stronger claims about future performance.


## Self-Check

* [x] Section 1 includes two research findings and a constructive methodology question for each.
* [x] Section 2 compares the original time-aware validation with a grouped-by-client validation.
* [x] The original time-aware results were MAE = 0.2620 and RMSE = 2.4575.
* [x] The grouped-by-client results were MAE = 0.5129 and RMSE = 47.0867.
* [x] The grouped split had 39 training clients and 10 testing clients, with 0 client overlap.
* [x] Section 3 audits the final feature set for label-derived, future/overlapping, and decision-derived leakage.
* [x] No known risky columns were used as model features.
* [x] The target timing audit identified gaps from 0 to 28 days, with a median gap of 1 day.
* [x] The audit identified 1,429 duplicate content-date combinations as a limitation of the current target construction.
* [x] Real failure examples were examined, including a largest absolute error of approximately 8,148.54 clicks.
* [x] Section 4 rewrites the model claim using careful language about measured results, directional decision-support, and generalization limits.
* [x] No client names, private queries, or sensitive information are included.
* [x] The notebook has been run from top to bottom and checked for errors.
* [x] The completed notebook is saved under `work/notebooks/w06_validation_audit.ipynb`.
* [ ] The notebook has been committed and pushed to the repository.
* [ ] The repository URL is ready to submit on the ML-09 assignment card.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.